# Complete 3D Molecular-Orbital Visualization

This notebook calculates **all canonical molecular orbitals** for one molecule, exports every orbital as its own Gaussian cube file, and provides an interactive 3D viewer.

Important: one cube file represents one molecular orbital. Orbitals are viewed one at a time because overlaying all orbital isosurfaces would hide their shapes. Blue and red show the two phases (signs) of the orbital, not electrical charge.

## 1. Imports and project path

In [ ]:
from pathlib import Path
import sys

# Locate the repository from the usual Jupyter starting directories.
cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / "vqe_cudaq",
    cwd / "BenchMark For Ground State" / "vqe_cudaq",
]
REPO_ROOT = next(
    (path for path in candidates if (path / "vqe_cudaq" / "__init__.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the vqe_cudaq repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, display

from vqe_cudaq.molecules import molecules
from vqe_cudaq.orbitals import (
    export_orbital_bundle,
    orbital_table,
    plot_orbital_energy_diagram,
    run_orbital_calculation,
    view_orbital_cube,
)

print("Repository:", REPO_ROOT)
print("Available molecules:", ", ".join(molecules))

## 2. Choose the molecule and calculation settings

Change `MOLECULE` to any exact name printed above. Cube grid 60 is good for inspection; use 80–120 for higher-quality figures, at the cost of larger files and longer generation time.

In [ ]:
MOLECULE = "Methylene"
BASIS = "cc-pVDZ"
METHOD = "hf"       # "hf" or "dft"
XC_FUNCTIONAL = "b3lyp"
CUBE_GRID = 60
ISOVALUE = 0.03
OUTPUT_DIR = REPO_ROOT / "molecular_orbitals"

if MOLECULE not in molecules:
    raise KeyError(f"Unknown molecule {MOLECULE!r}")

print(f"Selected: {MOLECULE} | {METHOD.upper()}/{BASIS}")

## 3. Run the SCF calculation

Closed-shell molecules use RHF/RKS. Open-shell molecules use ROHF/ROKS so that one consistent set of spatial orbitals is produced. Coordinate units are converted to Ångström automatically.

In [ ]:
calc = run_orbital_calculation(
    MOLECULE,
    molecules[MOLECULE],
    basis=BASIS,
    method=METHOD,
    xc=XC_FUNCTIONAL,
)

print("Converged:", calc.converged)
print(f"Total SCF energy: {calc.total_energy:.12f} Hartree")
print("Calculation:", f"{calc.method}/{calc.basis}")
print("Number of spatial molecular orbitals:", len(calc.mo_energy))

## 4. Examine every orbital's energy and occupation

`mo_index` is the zero-based index used in cube filenames and active-space work. `mo_number` is the corresponding one-based human-readable number.

In [ ]:
mo_table = orbital_table(calc)
display(mo_table)

occupied_rows = mo_table[mo_table["occupation"] > 1e-8]
virtual_rows = mo_table[mo_table["occupation"] <= 1e-8]
HOMO_INDEX = int(occupied_rows.iloc[-1]["mo_index"])
LUMO_INDEX = int(virtual_rows.iloc[0]["mo_index"]) if len(virtual_rows) else None
print("HOMO/SOMO index:", HOMO_INDEX)
print("LUMO index:", LUMO_INDEX)

## 5. Complete orbital-energy diagram

Unlike the default CLI frontier diagram, this cell explicitly includes every calculated MO.

In [ ]:
ALL_MO_INDICES = list(range(len(calc.mo_energy)))
fig, ax = plot_orbital_energy_diagram(
    calc,
    indices=ALL_MO_INDICES,
)
plt.show()

## 6. Export all orbitals

This generates one cube per MO, plus a Molden file containing all orbitals, a CSV table, and a PNG diagram. Cube generation is the slowest step. Generated data are ignored by Git.

In [ ]:
files = export_orbital_bundle(
    calc,
    output_dir=OUTPUT_DIR,
    indices=ALL_MO_INDICES,       # export every MO, including core orbitals
    grid=CUBE_GRID,
    export_cubes=True,
)

cube_by_index = dict(zip(files["selected_indices"], files["cubes"]))
total_cube_bytes = sum(path.stat().st_size for path in files["cubes"])

print("Output directory:", files["directory"])
print("All-orbital Molden file:", files["molden"])
print("Energy table:", files["table"])
print("Energy diagram:", files["diagram"])
print("Cube files generated:", len(files["cubes"]))
print(f"Total cube size: {total_cube_bytes / 1024**2:.1f} MiB")

## 7. Interactive 3D viewer for every MO

Select any orbital from the dropdown. The molecular geometry is shown with the positive phase in blue and negative phase in red. You can rotate, zoom, and pan with the mouse.

In [ ]:
def orbital_option_label(row):
    label = (
        f"MO {int(row.mo_index)} | {row.energy_ev:.3f} eV | "
        f"occupation={row.occupation:g}"
    )
    if row.frontier_label:
        label += f" | {row.frontier_label}"
    return label

orbital_options = [
    (orbital_option_label(row), int(row.mo_index))
    for row in mo_table.itertuples(index=False)
]

orbital_dropdown = widgets.Dropdown(
    options=orbital_options,
    value=HOMO_INDEX,
    description="Orbital:",
    layout=widgets.Layout(width="650px"),
)
isovalue_slider = widgets.FloatSlider(
    value=ISOVALUE,
    min=0.005,
    max=0.10,
    step=0.005,
    description="Isovalue:",
    continuous_update=False,
    readout_format=".3f",
)
viewer_output = widgets.Output()

def render_selected_orbital(change=None):
    mo_index = int(orbital_dropdown.value)
    row = mo_table.loc[mo_table["mo_index"] == mo_index].iloc[0]
    with viewer_output:
        viewer_output.clear_output(wait=True)
        display(HTML(
            f"<h3>{MOLECULE}: MO {mo_index}</h3>"
            f"<p>Energy = {row.energy_ev:.6f} eV; "
            f"occupation = {row.occupation:g}; "
            f"label = {row.frontier_label or '—'}</p>"
        ))
        view = view_orbital_cube(
            calc,
            cube_by_index[mo_index],
            isovalue=float(isovalue_slider.value),
            width=800,
            height=600,
        )
        view.show()

orbital_dropdown.observe(render_selected_orbital, names="value")
isovalue_slider.observe(render_selected_orbital, names="value")
display(widgets.VBox([orbital_dropdown, isovalue_slider, viewer_output]))
render_selected_orbital()

## 8. How to use these views for active-space selection

Start around the HOMO/SOMO and LUMO, then inspect neighboring orbitals. Keep orbitals that are near-degenerate or have chemically important character, and normally retain both members of a bonding/antibonding pair. For radicals, include the singly occupied orbital. Orbital energy alone is not a sufficient selection criterion.

The canonical MO indices are zero-based in this project. If all preceding orbitals are doubly occupied, a conventional contiguous active space from index `first_active` through `last_active` has `ncore = first_active` and `norb_cas = last_active - first_active + 1`; its electron count must be determined from the occupations in that interval.